In [1]:
'''
Waterfowl energetics model
Mike Mitchell - mmitchell@ducks.org - Ducks Unlimited

Duckdb
'''
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from datetime import datetime
import json
import matplotlib.pyplot as plt
import numpy as np
import scipy
import sys
import pandas as pd
import json, requests
import geopandas as gpd
import shapely
import duckdb
import re
import io
import base64
from shapely import wkt, wkb
import matplotlib.ticker as ticker
from datetime import datetime, timedelta
from shapely.geometry import Point
from scipy.spatial import cKDTree
from urllib.parse import urljoin
from scipy.interpolate import UnivariateSpline
import plotly.express as px
import plotly.io as pio
import warnings
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from IPython.display import Javascript
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# Suppress specific warning types
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.float_format', lambda x : "{:,.2f}".format(x))


loading_bar = widgets.Output()

header = widgets.HTML(
    value="<h2 style='margin-bottom: 20px;'>🦆 Waterfowl Energy Model Dashboard</h2><p>Adjust the parameters below and click <strong>Run Model</strong> to update the plots.</p>",
    layout=widgets.Layout(margin='10px 0px 20px 0px')
)
# Define widgets
start_date_widget = widgets.Text(
    value="Aug 1 2023",
    description='Start date:',
    placeholder='e.g., Aug 1 2023'
)

numofdays_widget = widgets.IntText(
    value=228,
    description='Days to run the model:',
    style={'description_width': 'initial'}
)
gooseeat_widget = widgets.Checkbox(
    value=True,
    description='Apply goose foraging'
)
removewater_widget = widgets.Checkbox(
    value=False,
    description='Remove open water'
)

customcurves_widget = widgets.Checkbox(
    value=True,
    description='Use Custom Curves'
)

smoothcrops_widget = widgets.Checkbox(
    value=True,
    description='Smooth habitat curves'
)

kcalperduck_widget = widgets.IntText(
    value=295,
    style={'description_width': 'initial'},
    description='Waterfowl daily kcal:'
)
kcalpergoose_widget = widgets.IntText(
    value=500,
    style={'description_width': 'initial'},
    description='Goose daily kcal:'
)
goosereduction_widget = widgets.IntText(
    value=50,
    style={'description_width': 'initial'},
    description='Reduce goose pop to this percentage:'
)
removeteal_widget = widgets.Checkbox(
    value=False,
    description='Remove Teal'
)
keepducks_widget = widgets.Textarea(
    value="AGWT, AMWI, BWTE, GADW, MALL, NOPI, NSHO, RNDU, WODU",
    description='Keep Ducks:',
    layout=widgets.Layout(width='100%', height='20px'),
    placeholder='Enter species codes separated by commas'
)
customcurvesdict_default = """{
    'Arkansas_mav_MALL': [0, 0.0, 0.014301824, 0.017315462,0.38, 0.53, 1, 0.77, 0.52, 0.12],
    'Arkansas_wg_MALL': [0, 0, 0.007728601,0.014762799,0.30,0.33,0.57,1.00,0.68,0.14,0],
    'Louisiana_mav_MALL': [0,0.04,0.52,0.80,1.00,0.51,0.20],
    'Mississippi_MALL':[0, 0,0.027916566, 0.056929146, 0.448400933, 0.111111111, 0.555555556, 1, 0.888888889, 0.444444444, 0.120744769, 0],
    'Tennessee_MALL':[0, 0, 0.2, 0.2 ,0.27,0.23,0.50,0.78,1.00,0.79,0.76,0.50,0.41]
}"""
customcurvesdict_default = customcurvesdict_default.replace("'", '"')
customcurvesdict_default = re.sub(r'(?<![\d])\.(\d+)', r'0.\1', customcurvesdict_default)
customcurvesdict_widget = widgets.Textarea(
    value=customcurvesdict_default,
    description='Waterfowl Curve change:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='200px'),
    placeholder='Enter dictionary as JSON'
)
#Curve of length = 8.  0=August, 7=March
goosecurve_defaults = """{
    'Arkansas_mav':{'pop':1238550,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Arkansas_wg':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Illinois':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Kentucky':{'pop':0,'curve':[0,0,.1,.3,1,1,.8,.2]},
    'Louisiana_mav':{'pop':164000,'curve':[0,0,0,.3,1,1,.5,.2]},
    'Louisiana_wg':{'pop':0,'curve':[0,0,0,.3,1,1,.5,.2]},
    'Mississippi':{'pop':270000,'curve':[0,0,0,.3,1,.3,.3,.2]},
    'Missouri':{'pop':0,'curve':[0,0,0,.3,1,.3,.3,.2]},
    'Oklahoma':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Tennessee':{'pop':52000,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Texas':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]}
    }"""
goosecurve_defaults = goosecurve_defaults.replace("'", '"')
goosecurve_defaults = re.sub(r'(?<![\d])\.(\d+)', r'0.\1', goosecurve_defaults)
goosereductionpct = 50 #28 percent of the values listed above will be used
goosecrops = ['corn', 'milo', 'sorghum', 'rice','soybeans']
goosecurvedict_widget = widgets.Textarea(
    value=goosecurve_defaults,
    description='Goose Curves:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%', height='220px'),
    placeholder='Enter dictionary as JSON'
)

run_button = widgets.Button(
    description="Run Model", 
    button_style='success',
    layout=widgets.Layout(margin='20px 0px 10px 0px')  # top right bottom left
)
settings_output = widgets.Output()
with settings_output:
    display(HTML("<br><br>"))
# Display the widgets
ui = widgets.VBox([header,
    start_date_widget,
    numofdays_widget,
    removewater_widget,
    kcalperduck_widget,
    customcurves_widget,
    #removeteal_widget,
    #keepducks_widget,
    customcurvesdict_widget,                   
    gooseeat_widget,
    kcalpergoose_widget,
    goosereduction_widget,
    goosecurvedict_widget,
    #smoothcrops_widget,

    run_button,
    settings_output                   
    ])

table_widget = widgets.Output()
chart_widget = widgets.Output()
endtable_widget = widgets.Output()
energy_widget = widgets.Output()
pop_widget = widgets.Output()
log_widget = widgets.Output()
holdlog = widgets.VBox([
    log_widget
])
tab = widgets.Tab(children=[ui, table_widget, chart_widget,endtable_widget, energy_widget,pop_widget,holdlog])
tab.set_title(0, 'Settings')
tab.set_title(1, 'Setup Tables')
tab.set_title(2, 'Setup Charts')
tab.set_title(3, 'Leftover Tables')
tab.set_title(4, 'Leftover Charts')
tab.set_title(5, 'Population curves')
tab.set_title(6, 'Log & Download')

download_button = widgets.Button(
    description="Download CSV",
    button_style='success',
    icon='download'
)
def create_download_link(df, filename="data.csv"):
    log_widget.clear_output()
    print('Download was clicked.  Please wait for the link to display (~20 seconds).')
    print('Once you click the download link it will take a second to package.')
    display(Javascript(f'console.log("Download was clicked")'))
    # Convert DataFrame to CSV in memory
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    b64 = base64.b64encode(csv_buffer.getvalue().encode()).decode()
    href = f'<a download="{filename}" href="data:text/csv;base64,{b64}">Click to download {filename}</a>'
    return href

 
main_layout = widgets.VBox([
    loading_bar,
    tab
])
display(main_layout)

#############################################################
# Waterfowl in this analysis
#keepducks = ['AGWT', 'AMWI', 'BWTE', 'GADW', 'MALL', 'NOPI', 'NSHO', 'RNDU', 'WODU']

baseaoiurl = 'https://giscog.blob.core.windows.net/waterfowlmodel/' # Data location
aois = ['ARmav', 'ARwg', 'KY', 'LAmav', 'LAwg', 'MO', 'MS', 'OK', 'TN', 'TX'] # used to read in files. example: baseaoiurl+aoi+'daily_obj3.csv'

'''
# habitat type: 
    'energy (dud)':{
        Lo: unharvested dud, 
        Hi: harvested dud}, 
    'habitat availability': [curve with values between 0 and 100], 
    'decomposition': percentage per day  1% = 1, not 0.01
'''
cropdict = {
    'aquaculture': {
        'energy':{
            'Lo':3, 
            'High':3
            },
        'availability': [34,94,100], 
        'decomp':0
        },
    'corn': {
        'energy':{
            'Lo':1130,
            'High':27717
            },
        'availability':[3,100,27], 
        'decomp':1.57
        },
    'emergentwetlands': {
        'energy':{
            'Lo':283,
            'High':283
            },
        'availability':[19,89,100], 
        'decomp':0.18
        },
    'hardwoods': {
        'energy':{
            'Lo':23, 
            'High':917
            },
        'availability':[2,89,100], 
        'decomp':0.0037
        },
    'millet': {
        'energy':{
            'Lo':2638,
            'High':4338
            },
        'availability':[46,100,95], 
        'decomp':0.64
        },
    'milo': {
        'energy':{
            'Lo':1171,
            'High':8305
            },
        'availability':[3,100,30], 
        'decomp':0.322
        },     
    'moistsoil': {
        'energy':{
            'Lo':247, 
            'High':3513
            },
        'availability':[19,89,100], 
        'decomp':0.18
        },             
    'openwater': {
        'energy':{
            'Lo':3, 
            'High':3
            },
        'availability':[100,100,100], 
        'decomp':0.18
        },  
    'rice': {
        'energy':{
            'Lo':1099,
            'High':18602
            },
        'availability':[9,100,39], 
        'decomp':0.213
        },  
    'sorghum': {
        'energy':{
            'Lo':1171, 
            'High':8305
            },
        'availability':[3,100,30], 
        'decomp':0.322
        },  
    'soybeans': {
        'energy':{
            'Lo':248, 
            'High':5389
            },
        'availability':[5,100,38],
        'decomp':1.9
        }, 
    'woodywetlands': {
        'energy':{
            'Lo':23, 
            'High':917
            },
        'availability':[2,89,100], 
        'decomp':0.0037
        },              
    'wrp': {
        'energy':{
            'Lo':147, 
            'High':147
            },
        'availability':[2,89,100], 
        'decomp':0.0037
        }
}

# Habitat curve should be in percentages and max shouldn't be more than 100.
cropcurvedata = {key: val['availability'] for key, val in cropdict.items()}
# Decomposition rates as a daily percent decay.  To be calculated by day as leftover = leftover * (100-decay)
cropdecomp = {key: val['decomp'] for key, val in cropdict.items()}


In [2]:

#button to run everything
# --- Callback function triggered by button ---
def on_button_clicked(b):
    print(main_layout.children)
    with loading_bar:
        pbar = tqdm(total=5, desc="Running model")
    chart_widget.clear_output()
    table_widget.clear_output()
    settings_output.clear_output()
    log_widget.clear_output()
    endtable_widget.clear_output()
    energy_widget.clear_output()
    pop_widget.clear_output()
    
    with settings_output:
        display(HTML("<h3 style='margin-top:20px;'><b>Running model with the following settings</b></h3>"))
        print("Start Date:", start_date_widget.value)
        print("Number of Days:", numofdays_widget.value)
        print("Kcal per Duck per Day:", kcalperduck_widget.value)
        #print("Smooth Crops:", smoothcrops_widget.value)
        print("Custom Curves:", customcurves_widget.value)
        print("Remove Water:", removewater_widget.value)
        #print("Remove Teal:", removeteal_widget.value)
        print("Goose forage:", gooseeat_widget.value)
        print("Goose reduction percent:", goosereduction_widget.value)
        display(HTML("<br><br>"))
    # Access widget values
    start_date = start_date_widget.value
    numofdays = numofdays_widget.value
    kcalperduck = kcalperduck_widget.value
    geese = gooseeat_widget.value
    kcalpergoose = kcalpergoose_widget.value
    smoothcrops = smoothcrops_widget.value
    customcurves = customcurves_widget.value
    removewater = removewater_widget.value
    removeteal = removeteal_widget.value
    goosereductionpct = goosereduction_widget.value
    keepducks = [s.strip() for s in keepducks_widget.value.split(',') if s.strip()]

    try:
        customcurvesdict = json.loads(customcurvesdict_widget.value)
    except json.JSONDecodeError as e:
        with log_widget:
            print("⚠️ Invalid waterfowl curve dictionary format. Please check your input.")
            print(e)
        customcurvesdict = {}

    try:
        goosecurve = json.loads(goosecurvedict_widget.value)
    except json.JSONDecodeError as e:
        with log_widget:
            print("⚠️ Invalid goose curve dictionary format. Please check your input.")
        goosecurve = {}        

    # Checks to make sure habitat curves have at least 3 values and they are not > 100%        
    for key,val in cropcurvedata.items():
        if max(val) >100:
            print('Value in {} has a value greater than 100'.foDemat(key))
        if len(val) <3:
            print('Not a great curve with < 3 values for {}'.format(key))

    if removewater:
        if 'OpenWater'in cropcurvedata:
            del cropcurvedata['OpenWater']            

    start_date = datetime.strptime(start_date, "%b %d %Y")
    date_labels = [(start_date + timedelta(days=i)).strftime("%b_%d")+' day: '+str(i+1) for i in range(numofdays)]

    popcurves = urljoin(baseaoiurl + '/','popcurve.parquet')
    energydataset = urljoin(baseaoiurl + '/','energy.parquet')            
    energycsvurl = urljoin(baseaoiurl + '/','4D_LMVJV-ST_Obj_FINAL_LONG2.csv')
    
    # Setup duckdb
    con = duckdb.connect()
    con.install_extension("spatial")
    con.load_extension("spatial")
    con.install_extension("azure")
    con.load_extension("azure")            

    # Flatten into a list of records
    records = []
    for crop, values in cropdict.items():
        record = {
            'Class': crop,
            'ValueLo': values['energy']['Lo'],
            'ValueHi': values['energy']['High'],
            'availability': values['availability'],
            'decomp': values['decomp']
        }
        records.append(record)

    # Convert to DataFrame
    df = pd.DataFrame(records)
    display(HTML("<br><br>"))
    with table_widget:
        display(Markdown("Habitat types, energetic values (DED), availability curves, and daily decomposition %"))
        display(df)            

    # Read in energy dataset
    with loading_bar:
        pbar.update(1)
    con.sql('''
    CREATE OR REPLACE TABLE lmvjvwaterfowlenergy AS SELECT * FROM read_parquet('{0}')
    '''.format(energydataset))
    if removewater:
        con.sql('''DELETE FROM lmvjvwaterfowlenergy WHERE Class = 'openwater';''')
    
   
    inenergy = con.sql('select * from lmvjvwaterfowlenergy').df()
    '''
    Prep energy layer so we can calculate total energy at the county level.  Remove NAN
    '''
    with loading_bar:
        pbar.update(2)

    with table_widget:
        display(Markdown('Acreage and Energetic stats'))
        print('Acres:', f"{int(inenergy['acres'].sum()):,}")
        display(inenergy.groupby(['statebcr'])['acres'].sum())
        inenergy['totalkcals'] = inenergy['acres']*inenergy['energyvalue']
        print('')
        print('kcals total:', f"{int(inenergy['totalkcals'].sum()):,}")
        display(inenergy.groupby('statebcr')['totalkcals'].sum())
        # Read population objectives.  These are DUDs
        print('Population objective stats')
        popobjtable = pd.read_csv(energycsvurl)
        popobjtable = popobjtable.rename(columns={'State':'state', 'LMVJV.80P.OBJ':'popobj80'})
        print('Sum population objective (DUD): {0:,.0f}'.format(popobjtable['popobj80'].sum()))
        popobjtable.groupby('state')['popobj80'].sum()
        print('\n')
    # Read waterfowl curves and adjust attributes to align with long term objectives by statebcr
    con.sql('''
    CREATE OR REPLACE TABLE curvetablein AS SELECT * FROM read_parquet('{0}')
    '''.format(popcurves))
   
    curvetable = con.sql('select * from curvetablein').df().drop(columns=['__index_level_0__'])
    with table_widget:
        print('curvetable length', len(curvetable))
    with loading_bar:
        pbar.update(3)
        pbar.set_description("Finalizing")
    
    ##### Setup Custom waterfowl curves #####
    with chart_widget:
        display(HTML("<h3 style='margin-top:20px;'><b>Population curve replacement</b></h3>"))
        for k, v in customcurvesdict.items():
            sp = k.rsplit("_",1)[1]
            st = k.rsplit("_",1)[0]
            if sp == 'Other':
                continue
            forreplace = np.interp(np.linspace(0, numofdays, numofdays), [round(p) for p in np.linspace(0, numofdays, len(v))], v)
            spmax = curvetable[(curvetable['species']==sp)&(curvetable['statebcr']==st)]['max'].iloc[0]
            #newcurve = forreplace*spmax

            x_interp = np.linspace(0, numofdays, numofdays)
            x_known = [round(p) for p in np.linspace(0, numofdays, len(v))]
            v_interp = forreplace

            # Smooth using UnivariateSpline
            spline = UnivariateSpline(x_interp, v_interp)
            spline.set_smoothing_factor(0.01)
            v_smooth = spline(x_interp)

            # Scale smoothed values to preserve the original max
            original_max = np.max(v_interp)
            smoothed_max = np.max(v_smooth)
            v_smooth = v_smooth * (original_max / smoothed_max) *spmax
            v_smooth = v_smooth.clip(min=0)
            bythesp = list(curvetable[(curvetable['species']==sp)&(curvetable['statebcr']==st)].drop(columns=['species', 'statebcr']).T[:-1].T.iloc[0])
            bythesp = bythesp[:numofdays]
            plt.plot(date_labels,bythesp, label='Original', alpha=0.5)
            plt.plot(date_labels, v_smooth, label='Replaced', linewidth=2)
            plt.legend()
            plt.title("Curve replacement for {0} in {1}".format(sp, st))
            plt.xticks(np.arange(0, numofdays+2, step=20), rotation='vertical')
            plt.xlabel("Date")
            plt.ylabel("# of birds")
            plt.show()
            # Replace the values
            curvetable.loc[(curvetable['species'] == sp) & (curvetable['statebcr'] == st), [str(i) for i in range(1, numofdays+1)]] = v_smooth

    # Clean and copy data
    curveforplot = curvetable.copy()
    colors=['red', 'black', 'blue', 'brown', 'green', 'pink', 'cyan', 'purple', 'orange', 'yellow', 'grey', 'lime']
    # Create one interactive plot per statebcr
    with chart_widget:
        display(HTML("<h3 style='margin-top:20px;'><b></b></h3>"))
        for stbcr in curveforplot['statebcr'].unique():
            ct = curveforplot[curveforplot['statebcr'] == stbcr]

            # Sum over species by day
            tmp = ct.drop(columns='max').groupby('species').sum().drop(columns='statebcr')
            tmp = tmp.reset_index()

            # Max values for horizontal lines
            maxsp = ct[['statebcr', 'max', 'species']].groupby('species').sum().drop(columns='statebcr')
            maxsp = maxsp.reset_index()

            # Prepare figure
            fig = go.Figure()

            for i, sp in enumerate(tmp['species']):
                y = tmp[tmp['species'] == sp].drop(columns=['species']).values.flatten()
                x = date_labels[:len(y)]  # Match length if needed

                max_y = maxsp[maxsp['species'] == sp]['max'].values
                if len(max_y) == 0:
                    continue

                fig.add_trace(go.Scatter(
                    x=x,
                    y=y,
                    mode='lines',
                    name=sp,
                    line=dict(color=colors[i % len(colors)])
                ))

                # Add dashed horizontal max line
                fig.add_trace(go.Scatter(
                    x=x,
                    y=[max_y[0]] * len(x),
                    mode='lines',
                    line=dict(dash='dash', color=colors[i % len(colors)]),
                    name=f"{sp} max",
                    showlegend=False
                ))

            fig.update_layout(
                title=f"Species population curves with max line for {stbcr.replace('_', ' ')}",
                xaxis_title="Day",
                yaxis_title="Population objective (DUD)",
                xaxis=dict(tickmode='linear', tick0=0, dtick=20),
                yaxis=dict(tickformat=","),
                legend=dict(x=1.05, y=1),
                height=600
            )

            fig.show()


    aucsp = {}
    with chart_widget:
        # Start figure
        fig = go.Figure()

        # Build interactive plot
        for st in curvetable['statebcr'].unique():
            query = pd.DataFrame(curvetable[curvetable['statebcr'] == st]).drop(columns=['statebcr', 'species', 'max'], axis=1).sum(axis=0).reset_index()
            query.columns = ['index', 'value']
            query['index'] = query['index'].apply(int)

            # Store AUC (area under curve)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=DeprecationWarning)
                auc = np.trapz(query['value'], query['index'])
            aucsp[st] = auc

            # Add line to figure
            fig.add_trace(go.Scatter(
                x=date_labels[:len(query)],
                y=query['value'],
                mode='lines',
                name=st,
                hovertemplate=f"<b>{st}</b><br>Day: %{{x}}<br>Birds: %{{y:,.0f}}<extra></extra>"
            ))

        # Final plot formatting
        fig.update_layout(
            title="Energy demand by state / BCR",
            xaxis_title="Day",
            yaxis_title="# of birds",
            xaxis=dict(tickmode='linear', tick0=0, dtick=20),
            yaxis=dict(tickformat=","),
            height=600,
            legend=dict(x=1.05, y=1)
        )

        fig.show()
    curvetable = curvetable.groupby(['statebcr']).sum().reset_index()#.drop(['Unnamed: 0'], axis=1)
    curvetable = curvetable.drop(columns='species')
    curvetable = curvetable.replace([np.inf, -np.inf], np.nan).fillna(0)
    curvetable = curvetable.drop(columns=['max'], axis=1)
    
    with chart_widget:
        totaldemand = curvetable.sum().reset_index().drop([0])
        totaldemand = totaldemand[:numofdays]
        plt.plot(date_labels, totaldemand[0]*kcalperduck)
        plt.xticks(np.arange(0, numofdays+2, step=20), rotation='vertical')
        plt.title('Total Demand')
        plt.xlabel('Day', labelpad=10)
        plt.ylabel('Energy (kcal)')
        plt.yticks(rotation=45)
        plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
        print(totaldemand[0].mean())    

        # Calculate habitat curves
        pio.renderers.default = 'notebook'  # or 'notebook_connected', or 'iframe_connected'

        habitatcurvedata = {}

        for sp in cropcurvedata.keys():
            # Interpolation to daily steps
            x_interp = np.linspace(1, numofdays, numofdays)
            x_known = [round(p) for p in np.linspace(1, numofdays, len(cropcurvedata[sp]))]
            v_interp = np.interp(x_interp, x_known, cropcurvedata[sp])

            if smoothcrops:
                spline = UnivariateSpline(x_interp, v_interp)
                spline.set_smoothing_factor(1000)
                v_smooth = spline(x_interp)
                v_smooth = v_smooth * (100.0 / np.max(v_smooth))
                v_smooth = v_smooth.clip(min=0)
                habitatcurvedata[sp] = v_smooth
            else:
                habitatcurvedata[sp] = v_interp

        # Convert to DataFrame (wide format)
        habitatcurve = pd.DataFrame.from_dict(habitatcurvedata).transpose().reset_index().rename(columns={'index':'CLASS'})
        habitatcurve.columns = ['CLASS'] + list(range(1, numofdays + 1))

        # Convert to long format for Plotly
        habitatcurve_long = habitatcurve.melt(id_vars='CLASS', var_name='Day', value_name='PercentHabitat')

        # Optional: Add a real date label column if needed
        # habitatcurve_long['Date'] = pd.to_datetime('2024-07-01') + pd.to_timedelta(habitatcurve_long['Day'] - 1, unit='D')

        # Plotly interactive line plot
        fig = px.line(
            habitatcurve_long,
            x='Day',
            y='PercentHabitat',
            color='CLASS',
            title='Habitat availability over time',
            labels={
                'Day': 'Day',
                'PercentHabitat': '% habitat available',
                'CLASS': 'Class'
            },
            hover_name='CLASS',
            hover_data={'PercentHabitat': ':.2f'}
        )

        fig.update_layout(
            xaxis=dict(tickmode='linear', tick0=0, dtick=20),
            yaxis=dict(tickformat=".0f"),
            height=600
        )

        fig.show()
    
    inenergy = inenergy.dropna(subset=['statebcr'])

    if geese:
        with log_widget:
            print('Using goose curves')
        with chart_widget:
            ############ PREP Goose data ############
            goosecurvekcal = pd.DataFrame()
            rows=[]
            for k, v in goosecurve.items():
                curve = goosecurve[k]['curve']
                spmax = goosecurve[k]['pop'] * goosereductionpct*0.01
                if spmax == 0:
                    continue
                x_interp = np.linspace(0, numofdays, numofdays)
                x_known = [round(p) for p in np.linspace(0, numofdays, len(v))]
                v_interp = np.interp(np.linspace(0, numofdays, numofdays), [round(p) for p in np.linspace(0, numofdays, len(curve))], curve)

                # Smooth using UnivariateSpline
                spline = UnivariateSpline(x_interp, v_interp)
                spline.set_smoothing_factor(0.01)
                v_smooth = spline(x_interp)

                # Scale smoothed values to preserve the original max
                original_max = np.max(v_interp)
                smoothed_max = np.max(v_smooth)
                v_smooth = v_smooth * (original_max / smoothed_max) * spmax
                v_smooth = v_smooth.clip(min=0)

                row = {'statebcr': k}
                row.update({str(i + 1): v for i, v in enumerate(v_smooth)})
                rows.append(row)

                plt.plot(date_labels, v_smooth, label='_Replaced', linewidth=2)
                plt.title("Goose curve for {0}".format(k))
                plt.xticks(np.arange(0, numofdays+2, step=20), rotation='vertical')
                plt.xlabel("Date")
                plt.ylabel("# of birds")
                plt.show()            
            goosecurvekcal = pd.DataFrame(rows)
    else:
        with log_widget:
            print('No goose curves')          
            
    with table_widget:  
        # Energy under the curve
        tmpx = totaldemand['index'].apply(int)
        tmpy = totaldemand[0].apply(int)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=DeprecationWarning)
            totaldemandarea = np.trapz(tmpy,tmpx)
        print('Demand (DUD) AuC:',f'{totaldemandarea:,.0f}')
        print('Demand (DUD) sum:', f'{tmpy.sum():,.0f}')
        print('Demand (kcal) sum:', f'{tmpy.sum()* kcalperduck:,.0f}')
        print('Energy supply (kcal) sum:',f'{round(inenergy.totalkcals).sum():,.0f}')
        print('Energy supply (DUD) sum:',f'{round(inenergy.totalkcals/kcalperduck).sum():,.0f}')
        print('############')
        print('Demand (DUD)')
        print(pd.DataFrame({'statebcr': list(aucsp.keys()), 'demand': list(aucsp.values())}))
        blah = inenergy.groupby(['statebcr', 'totalkcals'],as_index=False).sum()[['statebcr', 'totalkcals', 'acres']]
        blah['energy supply'] = blah['totalkcals']
        print('Energy supply (kcal)')
        print(blah.groupby('statebcr', as_index=False).sum()[['statebcr', 'energy supply']])

    # testing daily iteration and aggregation
    trackenergy = pd.DataFrame() # create empty dataframe to hold output by day
    energylayer = inenergy.copy().fillna(0)
    energylayer['unique'] = energylayer.index
    energylayer['vegenergyprevLo'] = 0
    energylayer['leftoverLo'] = 0
    energylayer['startacres'] = energylayer['acres'] 
    energylayer['reduceacres'] = 0
    energylayer['totalsupplyLo'] = energylayer['acres'] * energylayer['energyvalue']
    trackhideficit = pd.DataFrame()
    tracklodeficit = pd.DataFrame()
    pd.set_option('display.max_columns', None)
    
    with table_widget:
        for i in range(1,numofdays+1): #1 to numofdays
            energylayer['day'] = i
            # Get habitat availability based on habitat curve and calculate available acres
            hab = habitatcurve[['CLASS', i]] # select habitat availability curve for day by class
            energylayer = energylayer.merge(hab, on='CLASS', how='left') # merge habitat curve to the energy layer
            energylayer['habpct'] = energylayer[i]
            energylayer = energylayer.drop(i, axis=1) # Drop habitat percentage day column
            energylayer['acres'] = (energylayer['acres'] - energylayer['reduceacres']).clip(lower=0)
            energylayer['availacres'] = energylayer['acres'] * energylayer['habpct']*.01# calculate available acres which is acres of the energy polygon * habitat type availability for that day.

            # Calculate energy supply
            energylayer['vegenergyLo'] = energylayer['availacres'] * energylayer['energyvalue']   
            energylayer['reserveEnergyLo'] = energylayer['totalsupplyLo'] - energylayer['vegenergyLo']
            energylayer['reserveAcres'] = energylayer['acres'] - energylayer['availacres']
            energylayer['reservekcal'] = energylayer['reserveAcres'] * energylayer['energyvalue'] 
            energylayer['diffLo'] = (energylayer['vegenergyLo'] - energylayer['vegenergyprevLo']).clip(lower=0) # Energy supply includes the leftover energy from the day before plus the difference between todays supply energy and yesterdays.
            energylayer['vegenergyprevLo'] = energylayer['vegenergyLo']
            # Add supply from previous day
            #energylayer['supplyLo'] = energylayer['vegenergyLo']
            energylayer['supplyLo'] = energylayer['leftoverLo']  + energylayer['diffLo']

            # Proportion demand based on energy supply at the record level.
            filtersupplyLo = energylayer[energylayer['supplyLo'] >= 0]
            energylayerbystatebcr = filtersupplyLo[['statebcr','supplyLo']].groupby(['statebcr']).sum().rename(columns={'supplyLo':'supplyLoMax'})
            if 'supplyLoMax' in energylayer.columns:
                energylayer = energylayer.drop('supplyLoMax', axis=1)
            energylayer = energylayer.merge(energylayerbystatebcr, on='statebcr', how='left')
            #energylayer['pctdemand'] = energylayer['supplyLo']/energylayer['supplyLoMax']
            def calculate_pctdemand(group):
                if (group['leftoverLo'] <= 0).any():
                    total_supply = group['totalsupplyLo'].sum()
                    group['pctdemand'] = group['totalsupplyLo'] / total_supply if total_supply != 0 else 0
                else:
                    group['pctdemand'] = group['supplyLo'] / group['supplyLoMax']
                return group

            # Apply logic per statebcr
            energylayer = energylayer.groupby('statebcr', group_keys=False).apply(calculate_pctdemand)


            energylayer.loc[np.isnan(energylayer['pctdemand']),['pctdemand']] = 0
            popcurve = curvetable[['statebcr', str(i)]] # select demand for the day based on curve.

            energylayer = energylayer.merge(popcurve, on='statebcr', how='left') # merge demand for that day based on statebcr
            energylayer['demand'] = abs(energylayer['pctdemand']*energylayer[str(i)]*kcalperduck)
            energylayer = energylayer.drop(str(i), axis=1, errors='ignore')
            # Calculate leftover energy
            energylayer['leftoverLo'] = energylayer['supplyLo'] - energylayer['demand']

            # Calculate reduction in acres from goose forage
            if geese:
                goosepopcurve = goosecurvekcal[['statebcr', str(i)]]
                energylayer = energylayer.merge(goosepopcurve, on='statebcr', how='left') # merge demand for that day based on statebcr
                energylayer['goosedemandkcal'] = 0
                energylayer[str(i)] = energylayer[str(i)].fillna(0)
                energylayer[str(i)] = energylayer[str(i)].astype('int64')
                energylayer.loc[energylayer['CLASS'].isin(goosecrops), 'goosedemandkcal'] = energylayer[str(i)] * kcalpergoose
                energylayer['reduceacres'] = (energylayer['goosedemandkcal'] / energylayer['energyvalue']).clip(lower=0)
                energylayer = energylayer.drop(str(i), axis=1, errors='ignore')

            # Decomp  *** need to only factor in decomp is leftover is positive.
            energylayer.loc[energylayer['leftoverLo'] > 0, 'leftoverLo'] *= ((100 - energylayer['decomp']) * 0.01)

            trackenergy = pd.concat([trackenergy, energylayer])    
            tracklodeficit = pd.concat([tracklodeficit,energylayer[['day','statebcr','leftoverLo']].groupby(['day','statebcr']).sum().reset_index()])

    with endtable_widget:
        #print('day 1')
        #sample = trackenergy[trackenergy['day']==1]
        #printme = sample[['statebcr','CLASS', 'acres', 'availacres','supplyLo', 'totalsupplyLo', 'reserveEnergyLo','demand', 'leftoverLo']].groupby(['statebcr','CLASS']).sum()
        #with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
            #display(printme)
        #printme = printme.reset_index()

        sample = trackenergy[trackenergy['day']==numofdays]
        print('day {0}'.format(numofdays))
        printme = sample[['statebcr', 'acres', 'availacres','supplyLo', 'totalsupplyLo', 'reserveEnergyLo','demand', 'leftoverLo']].groupby(['statebcr']).sum()
        print('Values in kcal')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
            display(printme)
        print('\nThese ran out of energy')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
            display(printme[printme['leftoverLo']<0])    
        printme = printme.reset_index()

        sample = trackenergy[trackenergy['day']==numofdays]
        printme = sample[['statebcr', 'leftoverLo']].groupby(['statebcr']).sum()
        printme['leftoverLo'] = printme['leftoverLo']/kcalperduck
        print('Values in DED')
        print(printme.sum())
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
            display(printme)
        print('\nThese ran out of energy')
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
            display(printme[printme['leftoverLo']<0])    
        printme = printme.reset_index()

    # Filter out invalid statebcr
    tracklodeficit = tracklodeficit[tracklodeficit['statebcr'] != 0]

    with energy_widget:
        # Start interactive figure
        fig = go.Figure()

        # Add each state's line to the plot
        for st in tracklodeficit['statebcr'].unique():
            query = tracklodeficit[tracklodeficit['statebcr'] == st].drop(['statebcr'], axis=1)

            # Align query with date_labels if needed
            y_vals = query['leftoverLo'].values
            x_vals = date_labels[:len(y_vals)]

            fig.add_trace(go.Scatter(
                x=x_vals,
                y=y_vals,
                mode='lines',
                name=st,
                hovertemplate=f"<b>{st}</b><br>Day: %{{x}}<br>Leftover kcal: %{{y:,.0f}}<extra></extra>"
            ))

        # Final layout
        fig.update_layout(
            title="Leftover Low by statebcr",
            xaxis_title="Day",
            yaxis_title="kcal",
            xaxis=dict(tickmode='linear', tick0=0, dtick=20),
            yaxis=dict(tickformat=","),
            height=600,
            legend=dict(x=1.05, y=1),
            margin=dict(l=60, r=60, t=60, b=60)
        )

        fig.show()
        for stbcr in tracklodeficit['statebcr'].unique():
            query = tracklodeficit[tracklodeficit['statebcr']==stbcr].drop(['statebcr'],axis=1)
            plt.plot(query[['day']], query[['leftoverLo']])
            #plt.ticklabel_format(style='plain')
            plt.title('Leftover energy low for {0}'.format(stbcr))
            plt.xlabel('Day')
            plt.ylabel('kcal')
            plt.yticks(rotation=45)
            plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
            plt.show()

        pio.renderers.default = 'notebook'
        # Plot all habtiat types leftoverlo
        agg_df = (
            trackenergy
            .groupby(['statebcr', 'CLASS', 'day'], as_index=False)
            .agg({'leftoverLo': 'sum'})
        )
    with energy_widget:
        # Loop through each statebcr and generate an interactive plot
        for stbcr in agg_df['statebcr'].unique():
            query = agg_df[agg_df['statebcr'] == stbcr]

            fig = px.line(
                query,
                x='day',
                y='leftoverLo',
                color='CLASS',
                title=f'Leftover energy low for {stbcr}',
                labels={
                    'leftoverLo': 'kcal',
                    'day': 'Day',
                    'CLASS': 'Class'
                },
                hover_name='CLASS',
                hover_data={'leftoverLo': ':.0f', 'day': True}
            )

            fig.update_layout(
                yaxis_tickformat=',',
                yaxis_title='kcal',
                xaxis_title='Day',
                legend_title='Class',
                height=600
            )

            fig.show()
    with energy_widget:            
        plt.plot(list(range(1,numofdays+1)),trackenergy[['day','leftoverLo']].groupby(['day']).sum().reset_index()['leftoverLo'])
        plt.title('Leftover low energy over time')
        #plt.legend(['Low energy value'])
        plt.xticks(np.arange(0, numofdays+2, step=20), rotation='vertical')
        plt.xlabel('Day', labelpad=10)
        plt.ylabel('Energy (kcal)')
        plt.yticks(rotation=45)
        plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))


        plt.plot(list(range(1,numofdays+1)),trackenergy[['day','demand']].groupby(['day']).sum().reset_index()['demand'])
        plt.title('Energy demand over time')
        plt.xticks(np.arange(0, numofdays+2, step=20), rotation='vertical')
        plt.xlabel('Day', labelpad=10)
        plt.ylabel('Energy (kcal)')
        plt.yticks(rotation=45)
        plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

        plt.plot(list(range(1,numofdays+1)),trackenergy[['day','demand']].groupby(['day']).sum().reset_index()['demand'])
        plt.plot(list(range(1,numofdays+1)),trackenergy[['day','leftoverLo']].groupby(['day']).sum().reset_index()['leftoverLo'])
        plt.legend(['Energy demand', 'Energy supply'])
        plt.title('Log Energy supply and demand over time')
        plt.yscale('log')
        plt.xticks(np.arange(0, numofdays+2, step=20), rotation='vertical')
        plt.xlabel('Day', labelpad=10)
        plt.ylabel('Log energy (kcal)')
        trackenergy.groupby('statebcr')['supplyLoMax'].min()
    
    with pop_widget:
        # Loop through each unique statebcr and species combination
        for stbcr in curveforplot['statebcr'].unique():
            for sp in curveforplot['species'].unique():
                ct = curveforplot[(curveforplot['statebcr'] == stbcr) & (curveforplot['species'] == sp)]
                if ct.empty:
                    continue

                # Drop statebcr, group by species (redundant now), sum by day
                tmp = ct.drop(columns='max').drop(columns='statebcr')
                y = tmp.drop(columns='species').sum().values
                x = date_labels[:len(y)]  # Ensure lengths match

                # Get max value for dashed line
                max_val = ct['max'].sum()

                # Create figure
                fig = go.Figure()

                fig.add_trace(go.Scatter(
                    x=x,
                    y=y,
                    mode='lines',
                    name=sp,
                    line=dict(color='blue'),
                    hovertemplate=f"<b>{stbcr} - {sp}</b><br>Day: %{{x}}<br>Population: %{{y:,.0f}}<extra></extra>"
                ))

                # Add max horizontal dashed line
                fig.add_trace(go.Scatter(
                    x=x,
                    y=[max_val] * len(x),
                    mode='lines',
                    name='Max',
                    line=dict(color='red', dash='dash'),
                    hoverinfo='skip',
                    showlegend=False
                ))

                # Update layout
                fig.update_layout(
                    title=f"{sp} population curve in {stbcr.replace('_', ' ')}",
                    xaxis_title="Day",
                    yaxis_title="Population objective (DUD)",
                    xaxis=dict(tickmode='linear', dtick=20),
                    yaxis=dict(tickformat=","),
                    height=500,
                    margin=dict(l=60, r=60, t=60, b=60)
                )
                fig.show()
    
    #settings_output.clear_output()
    def on_download_click(b):
        with log_widget:
            print('clicked download on log')
            display(widgets.HTML(create_download_link(trackenergy)))

    download_button.on_click(on_download_click)     
    with settings_output:
        display(HTML("<h3 style='margin-top:20px;'><b>Model run complete</b></h3>"))
        display(HTML("<br><br>"))
        # Append download_button and download_output to main_layout
    holdlog.children = tuple(holdlog.children) + (download_button,)
      
    with loading_bar:
        pbar.update(4)
        pbar.set_description("Complete")
    
# --- Bind button click to the callback ---
run_button.on_click(on_button_clicked)